🧠 주관식 분석 문제
🟢 기본 (EDA 수준)

Q1
2023 → 2024 사이
👉 20대 순이동이 가장 많이 증가한 지역은 어디인가?
👉 그 증가량은 얼마인가?

Q2

2024 기준
👉 평균소득이 가장 높은 지역 + 연령대 조합은?

Q3

👉 고용률과 평균소득은 양의 상관관계가 있다고 볼 수 있는가?
👉 이유를 데이터 기반으로 설명하라.

🟡 중급 (분석 해석)
Q4

👉 수도권(서울 + 경기)과 지방(부산 + 전남)의
순이동 차이를 비교하고
👉 청년 집중 현상이 있다고 볼 수 있는지 서술하라.

Q5

👉 20대 vs 30대 중
👉 어느 연령대가 지역 이동에 더 민감한가?
(순이동 변동 기준)

🟠 심화 (실무 스타일)
Q6

👉 순이동을 종속변수로 보고
👉 영향을 줄 것 같은 변수 2개 선택
👉 이유 설명

(예: 소득, 고용률 등)

Q7

👉 전남이 순유출 지역인 이유를
데이터 기반 가설 2개 제시하라.

Q8

👉 만약 정책으로 청년 유입을 늘리려면
👉 어떤 지표를 먼저 개선해야 할까?
👉 데이터 근거 포함해서 서술

🧪 추가 (코딩 문제 느낌)
Q9

pivot table 만들어서
👉 region vs year
👉 순이동 합계 비교

Q10

👉 2024 기준
👉 순이동 TOP2 지역 구하고
👉 왜 그런지 해석

💡 실무 감각 문제 (진짜 회사에서 자주 나오는 스타일)
Q11

상사가 이렇게 물어봄:

"왜 수도권으로 청년이 몰리는지
데이터로 설명해봐"

👉 5줄 이내로 요약

In [ ]:
import pandas as pd

df = pd.read_csv("./01_test.csv")

df


In [ ]:
df.info()
df.describe()

In [ ]:
# Q1
# 2023 → 2024 사이
# 👉 20대 순이동이 가장 많이 증가한 지역은 어디인가?
# 👉 그 증가량은 얼마인가?

df_20 = df[df["age_group"] == "20s"]

pivot = df_20.pivot(index="region", columns="year", values="net_migration")

pivot["increase"] = pivot[2024] - pivot[2023]

region = pivot["increase"].idxmax()
increase_value = pivot["increase"].max()

print(region, increase_value)

In [ ]:
df_20s = df[df["age_group"] == "20s"]

g = df_20s.set_index(["region", "year"])["net_migration"].unstack()

g["increase"] = g[2024] - g[2023]

print(g["increase"].idxmax(), g["increase"].max())


In [ ]:
(
    df[df.age_group == "20s"]
    .pivot(index="region", columns="year", values="net_migration")
    .assign(increase=lambda x: x[2024] - x[2023])["increase"]
    .idxmax()
)


In [ ]:
df

In [ ]:
# Q2 2024 기준 평균소득이 가장 높은 지역 + 연령대 조합은?

# df_g = df.groupby(by="region")["avg_income"].sum().sort_values(ascending=False)

# df_r = df[df["region"] == "Seoul"]
# df_r.groupby(by="age_group")["avg_income"].sum().sort_values(ascending=False)
df_2024 = df[df["year"] == 2024]

result = df_2024.loc[df_2024["avg_income"].idxmax()]

print(result)


In [ ]:
df_2024 = df[df["year"] == 2024]

(df_2024.sort_values("avg_income", ascending=False).head(1))


In [ ]:
# Q3
# 👉 고용률과 평균소득은 양의 상관관계가 있다고 볼 수 있는가?
# 👉 이유를 데이터 기반으로 설명하라.

df.loc[:, ["avg_income", "employment_rate"]].sort_values(
    by="employment_rate", ascending=False
)

# 고용률이 높은수록 평균 소득이 높다.

In [ ]:
df[["avg_income", "employment_rate"]].corr()

In [ ]:
df.plot.scatter(x="employment_rate", y="avg_income")

In [ ]:
# 👉 분석할 때
# 가장 무서운 실수 1개만 고르면 뭐라고 생각해요?

# 1. 컬럼 의미 착각 (예: 비율을 합계로 더함)
# 2. 단위/스케일 혼동 (만원 vs 원, %, ‰)
# 3. 집계 수준 불일치 (개인 vs 지역 vs 기간)
# 4. 라벨 정의 오류 (타겟 생성 로직 버그)

In [ ]:
df

In [ ]:
# Q4
# 👉 수도권(서울 + 경기)과 지방(부산 + 전남)의 순이동 차이를 비교하고 청년 집중 현상이 있다고 볼 수 있는지 서술하라.

# 수도권 / 지방 구분 컬럼 생성
metro = ["Seoul", "Gyeonggi"]

df["area"] = df["region"].apply(lambda x: "Metro" if x in metro else "Non_metro")

# area 기준 순이동 합
result = df.groupby("area")["net_migration"].sum()

print(result)


In [ ]:
result = (
    df[df.year == 2024]
    .assign(area=lambda x: x.region.isin(["Seoul", "Gyeonggi"]))
    .assign(area=lambda x: x.area.map({True: "Metro", False: "Non_metro"}))
    .groupby("area")["net_migration"]
    .sum()
)

result


In [ ]:
# Q5
# 👉 20대 vs 30대 중
# 👉 어느 연령대가 지역 이동에 더 민감한가?
# (순이동 변동 기준)

pivot = df.pivot_table(
    index="age_group", columns="year", values="net_migration", aggfunc="sum"
)

pivot["change"] = abs(pivot[2024] - pivot[2023])

pivot


In [ ]:
df.groupby("age_group")["net_migration"].std()

In [ ]:
df

In [ ]:
# Q6
# 👉 순이동을 종속변수로 보고
# 👉 영향을 줄 것 같은 변수 2개 선택
# 👉 이유 설명
# (예: 소득, 고용률 등)

